# ML-10 — Action Playbook and Ranked Queue

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sohila-Khaled-Abbas/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

Generating the final ranked content refresh recommendation playbook with action labels.

## 1. Ranked Recommendation Playbook

*Mapping top scores to explicit editorial action labels.*

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import RandomForestClassifier

csv_path = "../../data/raw/content_refresh_anonymized.csv"
if not os.path.exists(csv_path):
    csv_path = "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(csv_path)
df["pos_clean"] = np.where(df["avg_position"] == 0, 50.0, df["avg_position"])
df["click_drop"] = np.maximum(0, df["clicks_prev_30d"] - df["clicks_last_30d"])
df["target_binary"] = (df["click_drop"] > df["click_drop"].median()).astype(int)

features = ["impressions_90d", "clicks_prev_30d", "pos_clean", "content_age_days", "ctr"]
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(df[features], df["target_binary"])

df["model_score"] = rf.predict_proba(df[features])[:, 1]
df["action_label"] = np.where(df["click_drop"] > 50, "REFRESH_AND_EXPAND_CONTENT", "OPTIMIZE_TITLE_AND_METAS")

ranked_playbook = df.sort_values(by="model_score", ascending=False)
print("=== TOP 10 RECOMMENDATIONS PLAYBOOK ===")
print(ranked_playbook[["content_id", "client_id", "model_score", "click_drop", "action_label"]].head(10).to_string(index=False))


=== TOP 10 RECOMMENDATIONS PLAYBOOK ===
          content_id         client_id  model_score  click_drop             action_label
content_920bb86da524 client_19581e27de     0.986273           1 OPTIMIZE_TITLE_AND_METAS
content_bc21baf71fb4 client_19581e27de     0.983512           1 OPTIMIZE_TITLE_AND_METAS
content_2557c1c418b6 client_19581e27de     0.982612           1 OPTIMIZE_TITLE_AND_METAS
content_4897e6088934 client_19581e27de     0.979967           1 OPTIMIZE_TITLE_AND_METAS
content_910077c1bba4 client_19581e27de     0.977430           1 OPTIMIZE_TITLE_AND_METAS
content_0e8a62788a4a client_19581e27de     0.977297           1 OPTIMIZE_TITLE_AND_METAS
content_2105b32488af client_9f14025af0     0.977153           1 OPTIMIZE_TITLE_AND_METAS
content_a760753ae6f1 client_19581e27de     0.976628           1 OPTIMIZE_TITLE_AND_METAS
content_9f787a5d8e6f client_f74efabef1     0.973168           1 OPTIMIZE_TITLE_AND_METAS
content_5303b57911a1 client_9f14025af0     0.972799           1 OPTIMI

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`